# SMALL — Standalone cross-task improvement add-in

This is a **true standalone notebook**. Open it in a fresh Colab runtime; do **not** append it to the main Small notebook.

It:
- mounts Drive and rebuilds only the lightweight GEAR context;
- reuses cached Small explicit features;
- reconstructs V1/MS OOF predictions directly from saved fold prediction files;
- exports Small OOF for the Nuclear standalone notebook;
- waits for the Nuclear export;
- tests Nuclear cross-task OOF probabilities, with EVO only as a control, using Dev-only MLP CV.

No PDB normalization or V1/MS retraining is required.

In [ ]:
# ============================================================
# STANDALONE BOOTSTRAP — fresh Colab runtime
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, sys, json, ast, time, subprocess
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import average_precision_score

BASE = Path('/content/drive/MyDrive/Protein-binding-Phi')

# Lightweight requirements only.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "numpy", "pandas", "scipy", "scikit-learn", "tqdm"],
    check=True
)

ULTIMATE_CANDIDATES = [
    BASE/'GEAR_ULTIMATE_OTHER_BINDINGS',
    BASE/'GEAR_ULTIMATE',
]
ULTIMATE_DIR = next(
    (p for p in ULTIMATE_CANDIDATES if (p/'gear_ultimate').exists()),
    None
)
if ULTIMATE_DIR is None:
    raise FileNotFoundError(
        'GEAR backend not found under '
        'GEAR_ULTIMATE_OTHER_BINDINGS or GEAR_ULTIMATE'
    )

req = ULTIMATE_DIR/'requirements.txt'
if req.exists():
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(req)],
        check=True
    )

sys.path.insert(0, str(ULTIMATE_DIR))

from gear_ultimate import gear_evolution as evo
from gear_ultimate import gear_suite as suite
from gear_ultimate import gear_reporting as reporting

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cuda':
    torch.set_float32_matmul_precision('high')

print('BASE:', BASE)
print('GEAR backend:', ULTIMATE_DIR)
print('DEVICE:', DEVICE)


Mounted at /content/drive
BASE: /content/drive/MyDrive/Protein-binding-Phi
GEAR backend: /content/drive/MyDrive/Protein-binding-Phi/GEAR_ULTIMATE_OTHER_BINDINGS
DEVICE: cuda


In [ ]:
# Standalone settings
# These notebooks are designed for TWO FRESH COLAB RUNTIMES.
# Launch Nuclear and Small at roughly the same time.
# Each exports its Dev OOF early, then waits for the counterpart.
print("Standalone add-in runtime started.")


Standalone add-in runtime started.


In [ ]:
# ============================================================
# DATA / OFFICIAL FOLDS / STORE HELPERS
# ============================================================
TRAIN_CSV = BASE/'data/development_set/full_grouped_train_binding_sites_df.csv'
TEST300_CSV = BASE/'data/development_set/full_grouped_test_binding_sites_df.csv'
TEST46_CSV = BASE/'data/independent_set/grouped_test_46_new_binding_sites.csv'

def parse_list(v):
    if isinstance(v, str):
        return list(ast.literal_eval(v))
    return list(v)

def load_df(path):
    df = pd.read_csv(path)
    df['prot_id'] = df['prot_id'].astype(str)
    df['sequence'] = df['sequence'].astype(str)
    for col in [
        'metal_binding', 'nuclear_binding',
        'small_binding', 'any_ligand_binding_sites'
    ]:
        if col in df.columns:
            df[col] = df[col].apply(parse_list)
    return df

raw_train = load_df(TRAIN_CSV)
test300_df = load_df(TEST300_CSV)
test46_df = load_df(TEST46_CSV)

BASELINE_WORK = BASE/'baseline_reproduction_master_v6'
BN24_CANDIDATES = [
    BASELINE_WORK/'repos'/'bindnode24',
    BASE/'repos'/'bindnode24',
]
BN24 = next((p for p in BN24_CANDIDATES if (p/'.git').exists()), None)

if BN24 is None:
    BN24 = Path('/content/bindnode24_standalone_addin')
    if not BN24.exists():
        subprocess.run(
            ['git','clone','-q','https://github.com/erckert/bindNode24.git',str(BN24)],
            check=True
        )

fold_root = BN24/'dataset'/'cv_splits'/'predicted_structures'
OFFICIAL_FOLDS = {}
official_order = []
seen = set()

for k in range(1,6):
    ids = [
        x.strip()
        for x in (fold_root/f'ids_split{k}.txt').read_text().splitlines()
        if x.strip()
    ]
    if seen.intersection(ids):
        raise RuntimeError(f'Fold overlap at fold {k}')
    OFFICIAL_FOLDS[k] = ids
    official_order.extend(ids)
    seen.update(ids)

train_lookup = raw_train.set_index('prot_id', drop=False)
missing = set(official_order) - set(raw_train.prot_id)
if missing:
    raise RuntimeError(f'Missing official Dev IDs: {sorted(missing)[:20]}')

train_df = train_lookup.loc[official_order].reset_index(drop=True)

assert len(train_df) == 1010
assert len(test300_df) == 300
assert len(test46_df) == 46

SPLIT_DF = {
    'DevSet1010': train_df,
    'TestSet300': test300_df,
    'TestSetNew46': test46_df,
}
STORE_KEY = {
    'DevSet1010': 'train',
    'TestSet300': 'test300',
    'TestSetNew46': 'test46',
}

def get_record(store, pid):
    pid = str(pid)
    if pid in store:
        return store[pid]
    raise KeyError(pid)

def get_block(store, pid, block):
    rec = get_record(store, pid)

    if isinstance(rec, dict):
        if block in rec:
            return np.asarray(rec[block])
        if isinstance(rec.get('features'), dict) and block in rec['features']:
            return np.asarray(rec['features'][block])

    if hasattr(rec, block):
        return np.asarray(getattr(rec, block))

    for attr in ('features','feature_dict','blocks'):
        d = getattr(rec, attr, None)
        if isinstance(d, dict) and block in d:
            return np.asarray(d[block])

    raise KeyError(f'{pid}/{block}')

def set_block(store, pid, block, arr):
    rec = get_record(store, pid)
    arr = np.asarray(arr, dtype=np.float32)

    if isinstance(rec, dict):
        if isinstance(rec.get('features'), dict):
            rec['features'][block] = arr
        else:
            rec[block] = arr
        return

    for attr in ('features','feature_dict','blocks'):
        d = getattr(rec, attr, None)
        if isinstance(d, dict):
            d[block] = arr
            return

    setattr(rec, block, arr)

def reconstruct_oof_from_fold_predictions(pred_dir, label_col):
    """
    Saved foldX_val.npy was produced on:
      va_df = train_df[train_df.prot_id.isin(val_set)]
    preserving train_df order.
    """
    offsets = {}
    cursor = 0
    for r in train_df.itertuples():
        L = len(r.sequence)
        offsets[str(r.prot_id)] = (cursor, cursor+L)
        cursor += L

    oof = np.full(cursor, np.nan, dtype=np.float32)

    for fold, val_ids in OFFICIAL_FOLDS.items():
        val_set = set(val_ids)
        va_df = train_df[train_df.prot_id.isin(val_set)].copy()

        fp = Path(pred_dir)/f'fold{fold}_val.npy'
        if not fp.exists():
            raise FileNotFoundError(fp)

        pred = np.load(fp).astype(np.float32)
        expected = int(va_df.sequence.str.len().sum())

        if pred.shape != (expected,):
            raise RuntimeError(
                f'{fp}: shape {pred.shape}, expected {(expected,)}'
            )

        pos = 0
        for r in va_df.itertuples():
            L = len(r.sequence)
            a,b = offsets[str(r.prot_id)]
            oof[a:b] = pred[pos:pos+L]
            pos += L

    if not np.all(np.isfinite(oof)):
        raise RuntimeError('Incomplete reconstructed OOF')

    y = np.concatenate(
        train_df[label_col].apply(
            lambda x: np.asarray(x, dtype=np.int8)
        ).tolist()
    )

    return oof, y

def official_fold_indices():
    offsets = {}
    cursor = 0
    for r in train_df.itertuples():
        L = len(r.sequence)
        offsets[str(r.prot_id)] = (cursor,cursor+L)
        cursor += L

    out = {}
    for fold, ids in OFFICIAL_FOLDS.items():
        z = []
        for pid in ids:
            a,b = offsets[str(pid)]
            z.extend(range(a,b))
        out[fold] = np.asarray(z, dtype=np.int64)
    return out

FOLD_IDX = official_fold_indices()

def weight_search(y, p1, p2):
    rows = []
    for w in np.linspace(0,1,21):
        p = w*p1 + (1-w)*p2
        aps = [
            average_precision_score(y[idx], p[idx])
            for _,idx in sorted(FOLD_IDX.items())
        ]
        rows.append({
            'w_V1': float(w),
            'w_MS': float(1-w),
            'CV_AUPRC_mean': float(np.mean(aps)),
            'CV_AUPRC_std': float(np.std(aps, ddof=1)),
            'OOF_AUPRC': float(average_precision_score(y,p)),
            **{f'fold{i+1}_AUPRC': float(v) for i,v in enumerate(aps)}
        })
    return pd.DataFrame(rows).sort_values(
        ['CV_AUPRC_mean','w_V1'],
        ascending=[False,False]
    ).reset_index(drop=True)

print('Dataset/fold bootstrap: PASS')


Dataset/fold bootstrap: PASS


In [ ]:
# ============================================================
# SMALL STANDALONE CONTEXT + CACHED EXPLICIT BLOCKS + EVO CTRL
# ============================================================
ADDIN_ROOT = BASE/'results'/'ADDIN_STANDALONE_SMALL_XTASK'
ADDIN_ROOT.mkdir(parents=True, exist_ok=True)

XTASK_ROOT = BASE/'results'/'CROSSTASK_EXPORTS'
XTASK_ROOT.mkdir(parents=True, exist_ok=True)

ctx = suite.prepare_context(
    base=BASE,
    run_version='standalone_small_addin_context',
    label_column='small_binding',
    models=('MLP',),
    ensure_base=True,
    add_mech=True,
    add_multi=True,
    add_seqctx=True,
    esm_batch_size=8,
    device=DEVICE,
)

for split, df in SPLIT_DF.items():
    store_ids = set(map(str, ctx.stores[STORE_KEY[split]].keys()))
    csv_ids = set(df.prot_id.astype(str))
    if store_ids != csv_ids:
        raise RuntimeError(f'{split}: store/CSV mismatch')

# Attach cached small explicit blocks directly. No PDB rebuild.
FEATURE_CACHE = BASE/'cache'/'small_ultimate_v1'/'small_explicit_v1'
NEW_BLOCKS = {
    'SM_SEQCHEM32':32,
    'SM_POCKET3D44':44,
    'SM_LIGCHEM20':20,
    'SM_TOPO16':16,
    'SM_AROMSTACK12':12,
    'SM_PROTCTX8':8,
}

for split, df in SPLIT_DF.items():
    store = ctx.stores[STORE_KEY[split]]
    outdir = FEATURE_CACHE/split

    for r in df.itertuples():
        fp = outdir/f'{r.prot_id}.npz'
        if not fp.exists():
            raise FileNotFoundError(
                f'Missing cached small feature file: {fp}'
            )
        z = np.load(fp)
        for k,d in NEW_BLOCKS.items():
            a = z[k].astype(np.float32, copy=False)
            if a.shape != (len(r.sequence), d):
                raise RuntimeError(f'{split}/{r.prot_id}/{k}: {a.shape}')
            set_block(store, r.prot_id, k, a)

print('Cached small explicit blocks attached.')

# EVO is only a control for Small.
EVO_AVAILABLE = False
try:
    coverage = evo.attach_evo(
        ctx.stores,
        BASE,
        splits=('train','test300','test46'),
        require_msa=True,
    )
    display(coverage)
    EVO_AVAILABLE = True
    print('EVO attached as control.')
except Exception as e:
    print('EVO unavailable -> skip control.')
    print(repr(e))

display(reporting.dimension_table(ctx.stores['train']))


[DevSet1010] Reused ESM cache: DevSet1010_ESM2_t33_mean_last2_1280_fp16.pkl
[DevSet1010] Reused structure cache: DevSet1010_spatial_local_v2.pkl
[TestSet300] Reused ESM cache: TestSet300_ESM2_t33_mean_last2_1280_fp16.pkl
[TestSet300] Reused structure cache: TestSet300_spatial_local_v2.pkl
[TestSetNew46] Reused ESM cache: TestSetNew46_ESM2_t33_mean_last2_1280_fp16.pkl
[TestSetNew46] Reused structure cache: TestSetNew46_spatial_local_v2.pkl
[DevSet1010] Reused ATOM cache: DevSet1010_atomic_coordination_v1.pkl
[DevSet1010] Reused CHEM cache: DevSet1010_chemistry_v1_24d.pkl
[TestSet300] Reused ATOM cache: TestSet300_atomic_coordination_v1.pkl
[TestSet300] Reused CHEM cache: TestSet300_chemistry_v1_24d.pkl
[TestSetNew46] Reused ATOM cache: TestSetNew46_atomic_coordination_v1.pkl
[TestSetNew46] Reused CHEM cache: TestSetNew46_chemistry_v1_24d.pkl
[DevSet1010] Reused MECH1 cache: DevSet1010_mechanistic_v1.pkl
[DevSet1010] Reused MULTI2 cache: DevSet1010_multisignal_v2.pkl
[TestSet300] Reused 

EVO cache:   0%|          | 0/1353 [00:00<?, ?it/s]

,split,protein,msa_depth,has_real_msa
0,train,A0A0F7RDM3,977,True
1,train,A0A0F7RHX8,1024,True
2,train,A0A0H2W6Y8,1023,True
3,train,A0A2B6C3P9,1024,True
4,train,A0A384LKY8,542,True
...,...,...,...,...
1351,test46,Q9KDJ7,1023,True
1352,test46,Q9LFM3,1018,True
1353,test46,Q9SJ89,1022,True
1354,test46,U2EQ00,1021,True


EVO attached as control.


,block,dim
0,AA,20
6,ACCESS,10
4,ATOM,15
5,CHEM,24
9,CONF,4
11,ELEC3D,8
8,ENV,8
1,ESM,1280
22,EVO,48
3,LOCAL,8


In [ ]:
# ============================================================
# RECONSTRUCT SMALL V1/MS OOF FROM SAVED FOLD PREDICTIONS
# EXPORT FOR NUCLEAR STANDALONE NOTEBOOK
# ============================================================
V1_PRED_DIR = (
    BASE/'results'/'SMALL_ULTIMATE_V1_DATA_DRIVEN_POCKET'
    /'pocket_states'/'SM_PocketHG_Attention'
)

MS_PRED_DIR = (
    BASE/'results'/'SMALL_V1_ATTENTION_MULTISCALE_POCKETS'
    /'states'/'SM_PocketHG_Attention_MS'
)

V1_OOF, Y_DEV = reconstruct_oof_from_fold_predictions(
    V1_PRED_DIR, 'small_binding'
)
MS_OOF, Y2 = reconstruct_oof_from_fold_predictions(
    MS_PRED_DIR, 'small_binding'
)
assert np.array_equal(Y_DEV, Y2)

ws = weight_search(Y_DEV, V1_OOF, MS_OOF)
display(ws.head(10))
ws.to_csv(ADDIN_ROOT/'DEV_WEIGHT_SEARCH.csv', index=False)

best = ws.iloc[0]
W_V1 = float(best.w_V1)
W_MS = float(best.w_MS)
FINAL_OOF = W_V1*V1_OOF + W_MS*MS_OOF

print('Locked weights:', W_V1, W_MS)
print('Dev mean AUPRC:', float(best.CV_AUPRC_mean))

rows = []
pos = 0
for r in train_df.itertuples():
    L = len(r.sequence)
    y = np.asarray(r.small_binding, dtype=np.int8)
    for i in range(L):
        rows.append({
            'dataset':'DevSet1010',
            'prot_id':str(r.prot_id),
            'residue_index':i,
            'y_small':int(y[i]),
            'prob_FINAL':float(FINAL_OOF[pos+i]),
            'prob_V1':float(V1_OOF[pos+i]),
            'prob_MS':float(MS_OOF[pos+i]),
        })
    pos += L

export = pd.DataFrame(rows)
export_fp = XTASK_ROOT/'SMALL_DEV_OOF.csv.gz'
tmp_fp = XTASK_ROOT/'SMALL_DEV_OOF.csv.gz.tmp'
export.to_csv(tmp_fp, index=False)
os.replace(tmp_fp, export_fp)

print('Small OOF export:', export_fp)


,w_V1,w_MS,CV_AUPRC_mean,CV_AUPRC_std,OOF_AUPRC,fold1_AUPRC,fold2_AUPRC,fold3_AUPRC,fold4_AUPRC,fold5_AUPRC
0,0.60,0.40,0.409283,0.035424,0.395044,0.412509,0.458149,0.374628,0.375385,0.425746
1,0.55,0.45,0.409256,0.035038,0.395299,0.412701,0.457238,0.374688,0.375733,0.425922
2,0.65,0.35,0.409065,0.035797,0.394464,0.412071,0.458758,0.374274,0.374847,0.425376
3,0.50,0.50,0.409037,0.034590,0.395260,0.412780,0.456011,0.374847,0.375676,0.425870
4,0.45,0.55,0.408678,0.034145,0.394942,0.412877,0.454558,0.374607,0.375656,0.425691
5,0.70,0.30,0.408658,0.036168,0.393617,0.411329,0.459137,0.373738,0.374131,0.424957
6,0.75,0.25,0.408066,0.036586,0.392425,0.410514,0.459329,0.372861,0.373225,0.424402
7,0.40,0.60,0.408055,0.033690,0.394334,0.412767,0.452773,0.374213,0.375246,0.425273
8,0.35,0.65,0.407274,0.033119,0.393408,0.412503,0.450695,0.373863,0.374732,0.424580
9,0.80,0.20,0.407190,0.037073,0.390902,0.409508,0.459382,0.371795,0.371819,0.423447


Locked weights: 0.6000000000000001 0.3999999999999999
Dev mean AUPRC: 0.4092833208152622
Small OOF export: /content/drive/MyDrive/Protein-binding-Phi/results/CROSSTASK_EXPORTS/SMALL_DEV_OOF.csv.gz


In [ ]:
# ============================================================
# FIX SMALL OOF EXPORT — WRITE REAL GZIP
# ============================================================
from pathlib import Path
import os

XTASK_ROOT = BASE / 'results' / 'CROSSTASK_EXPORTS'
export_fp = XTASK_ROOT / 'SMALL_DEV_OOF.csv.gz'
tmp_fp = XTASK_ROOT / 'SMALL_DEV_OOF.tmp.csv.gz'

# `export` đã được tạo ở cell OOF trước đó
assert 'export' in globals(), "Need the previous OOF export cell state."

export.to_csv(
    tmp_fp,
    index=False,
    compression='gzip'
)

os.replace(
    tmp_fp,
    export_fp
)

# verify
import pandas as pd

_check = pd.read_csv(export_fp)

print("FIXED:", export_fp)
print("rows:", len(_check))
print("columns:", _check.columns.tolist())
print(_check.head())

FIXED: /content/drive/MyDrive/Protein-binding-Phi/results/CROSSTASK_EXPORTS/SMALL_DEV_OOF.csv.gz
rows: 170181
columns: ['dataset', 'prot_id', 'residue_index', 'y_small', 'prob_FINAL', 'prob_V1', 'prob_MS']
      dataset prot_id  residue_index  y_small  prob_FINAL   prob_V1   prob_MS
0  DevSet1010  Q5LL55              0        0    0.003769  0.002100  0.006272
1  DevSet1010  Q5LL55              1        0    0.004801  0.002372  0.008445
2  DevSet1010  Q5LL55              2        0    0.064171  0.086928  0.030034
3  DevSet1010  Q5LL55              3        0    0.003763  0.001797  0.006711
4  DevSet1010  Q5LL55              4        0    0.013890  0.007557  0.023390


In [ ]:
# ============================================================
# WAIT FOR NUCLEAR EXPORT -> ATTACH X-TASK -> SCREEN
# ============================================================
COUNTERPART = XTASK_ROOT/'NUCLEAR_DEV_OOF.csv.gz'
WAIT_SECONDS = 900
POLL_SECONDS = 10

t0 = time.time()
while not COUNTERPART.exists() and (time.time()-t0) < WAIT_SECONDS:
    print(
        f'Waiting for Nuclear export... '
        f'{int(time.time()-t0)}s / {WAIT_SECONDS}s',
        end='\r'
    )
    time.sleep(POLL_SECONDS)

XTASK_NUCLEAR_AVAILABLE = COUNTERPART.exists()
print('\nNuclear cross-task export available:', XTASK_NUCLEAR_AVAILABLE)

if XTASK_NUCLEAR_AVAILABLE:
    xdf = pd.read_csv(COUNTERPART)
    xdf['prot_id'] = xdf['prot_id'].astype(str)

    store = ctx.stores['train']
    for r in train_df.itertuples():
        pid = str(r.prot_id)
        sub = xdf[xdf.prot_id.eq(pid)].sort_values('residue_index')
        if len(sub) != len(r.sequence):
            raise RuntimeError(f'{pid}: cross-task length mismatch')
        arr = sub[['prob_FINAL','prob_V1','prob_MS']].to_numpy(np.float32)
        set_block(store, pid, 'XTASK_NUCLEAR3', arr)

BASE_BLOCKS = [
    'ESM',
    'ACCESS',
    'SM_POCKET3D44',
    'SM_TOPO16',
]

GRID = {
    'S_BASE_POCKET_GEOMETRY': BASE_BLOCKS,
}

if XTASK_NUCLEAR_AVAILABLE:
    GRID['S_BASE_PLUS_XTASK_NUCLEAR'] = BASE_BLOCKS + ['XTASK_NUCLEAR3']

if EVO_AVAILABLE:
    GRID['S_BASE_PLUS_EVO_CONTROL'] = BASE_BLOCKS + ['EVO']
    if XTASK_NUCLEAR_AVAILABLE:
        GRID['S_BASE_PLUS_EVO_XTASK_NUCLEAR'] = (
            BASE_BLOCKS + ['EVO','XTASK_NUCLEAR3']
        )

print('Screening:')
for k,v in GRID.items():
    print(' ',k,'=',v)

cv, folds = suite.run_dev_cv_only(
    ctx,
    GRID,
    models=('MLP',),
    run_version='standalone_small_xtask_screen',
)

base = float(
    cv.loc[
        cv.representation.eq('S_BASE_POCKET_GEOMETRY'),
        'CV_AUPRC'
    ].iloc[0]
)

cv = cv.copy()
cv['delta_vs_base'] = cv['CV_AUPRC'] - base
cv = cv.sort_values('CV_AUPRC', ascending=False).reset_index(drop=True)

display(cv)
display(folds)

cv.to_csv(ADDIN_ROOT/'REPRESENTATION_SCREEN.csv', index=False)
folds.to_csv(ADDIN_ROOT/'REPRESENTATION_SCREEN_FOLDS.csv', index=False)

best = cv.iloc[0]
decision = {
    'base_dev_auprc': base,
    'best_representation': str(best.representation),
    'best_dev_auprc': float(best.CV_AUPRC),
    'gain_vs_base': float(best.delta_vs_base),
    'promote_if_gain_ge_0.002': bool(best.delta_vs_base >= 0.002),
    'EVO_control_available': bool(EVO_AVAILABLE),
    'XTASK_nuclear_available': bool(XTASK_NUCLEAR_AVAILABLE),
}
(ADDIN_ROOT/'DECISION.json').write_text(json.dumps(decision, indent=2))

print('\nFINAL DECISION')
print(json.dumps(decision, indent=2))
print('Saved:', ADDIN_ROOT)



Nuclear cross-task export available: True
Screening:
  S_BASE_POCKET_GEOMETRY = ['ESM', 'ACCESS', 'SM_POCKET3D44', 'SM_TOPO16']
  S_BASE_PLUS_XTASK_NUCLEAR = ['ESM', 'ACCESS', 'SM_POCKET3D44', 'SM_TOPO16', 'XTASK_NUCLEAR3']
  S_BASE_PLUS_EVO_CONTROL = ['ESM', 'ACCESS', 'SM_POCKET3D44', 'SM_TOPO16', 'EVO']
  S_BASE_PLUS_EVO_XTASK_NUCLEAR = ['ESM', 'ACCESS', 'SM_POCKET3D44', 'SM_TOPO16', 'EVO', 'XTASK_NUCLEAR3']
REUSE MLP S_BASE_POCKET_GEOMETRY 1
REUSE MLP S_BASE_POCKET_GEOMETRY 2
REUSE MLP S_BASE_POCKET_GEOMETRY 3
REUSE MLP S_BASE_POCKET_GEOMETRY 4
REUSE MLP S_BASE_POCKET_GEOMETRY 5
REUSE MLP S_BASE_PLUS_XTASK_NUCLEAR 1
REUSE MLP S_BASE_PLUS_XTASK_NUCLEAR 2
REUSE MLP S_BASE_PLUS_XTASK_NUCLEAR 3
REUSE MLP S_BASE_PLUS_XTASK_NUCLEAR 4
REUSE MLP S_BASE_PLUS_XTASK_NUCLEAR 5
REUSE MLP S_BASE_PLUS_EVO_CONTROL 1
REUSE MLP S_BASE_PLUS_EVO_CONTROL 2
REUSE MLP S_BASE_PLUS_EVO_CONTROL 3
REUSE MLP S_BASE_PLUS_EVO_CONTROL 4
REUSE MLP S_BASE_PLUS_EVO_CONTROL 5
REUSE MLP S_BASE_PLUS_EVO_XTASK_NUCLEAR 

,model,representation,CV_AUPRC,CV_AUPRC_SD,CV_AUROC,delta_vs_base
0,MLP,S_BASE_POCKET_GEOMETRY,0.377708,0.033064,0.849750,0.000000
1,MLP,S_BASE_PLUS_XTASK_NUCLEAR,0.376403,0.032913,0.860940,-0.001305
2,MLP,S_BASE_PLUS_EVO_CONTROL,0.374507,0.028909,0.854665,-0.003201
3,MLP,S_BASE_PLUS_EVO_XTASK_NUCLEAR,0.367360,0.033445,0.863022,-0.010349


,model,representation,fold,AUPRC,AUROC,F1@0.5,MCC@0.5
0,MLP,S_BASE_POCKET_GEOMETRY,1,0.384631,0.863065,0.386779,0.367907
1,MLP,S_BASE_POCKET_GEOMETRY,2,0.423221,0.861593,0.415854,0.387586
2,MLP,S_BASE_POCKET_GEOMETRY,3,0.339725,0.835284,0.362764,0.337324
3,MLP,S_BASE_POCKET_GEOMETRY,4,0.351688,0.842518,0.367197,0.342158
4,MLP,S_BASE_POCKET_GEOMETRY,5,0.389275,0.846291,0.395137,0.363943
5,MLP,S_BASE_PLUS_XTASK_NUCLEAR,1,0.390429,0.868881,0.353081,0.345072
6,MLP,S_BASE_PLUS_XTASK_NUCLEAR,2,0.421287,0.882038,0.304010,0.326416
7,MLP,S_BASE_PLUS_XTASK_NUCLEAR,3,0.335652,0.854736,0.274135,0.291010
8,MLP,S_BASE_PLUS_XTASK_NUCLEAR,4,0.355152,0.841025,0.361930,0.340880
9,MLP,S_BASE_PLUS_XTASK_NUCLEAR,5,0.379494,0.858020,0.314488,0.319707



FINAL DECISION
{
  "base_dev_auprc": 0.3777080103628627,
  "best_representation": "S_BASE_POCKET_GEOMETRY",
  "best_dev_auprc": 0.3777080103628627,
  "gain_vs_base": 0.0,
  "promote_if_gain_ge_0.002": false,
  "EVO_control_available": true,
  "XTASK_nuclear_available": true
}
Saved: /content/drive/MyDrive/Protein-binding-Phi/results/ADDIN_STANDALONE_SMALL_XTASK


In [ ]:
from pathlib import Path
import gzip, time, json
import numpy as np
import pandas as pd

print('='*100)
print('SMALL CLEAN V3 — TRUE NUCLEAR -> SMALL CROSS-TASK SCREEN')
print('='*100)

COUNTERPART = XTASK_ROOT / 'NUCLEAR_DEV_OOF.csv.gz'
WAIT_SECONDS = 300
POLL_SECONDS = 5

t0 = time.time()
while not COUNTERPART.exists() and (time.time() - t0) < WAIT_SECONDS:
    print(f'Waiting for Nuclear export... {int(time.time()-t0)}s', end='\r')
    time.sleep(POLL_SECONDS)

if not COUNTERPART.exists():
    raise FileNotFoundError(COUNTERPART)

def _read_maybe_gzip_csv(fp):
    with open(fp, 'rb') as f:
        magic = f.read(2)

    if magic == b'\x1f\x8b':
        return pd.read_csv(fp)

    print(
        'WARNING: counterpart has .gz suffix but is plain CSV; '
        'reading safely as plain CSV.'
    )
    return pd.read_csv(fp, compression=None)

xdf = _read_maybe_gzip_csv(COUNTERPART)

xdf['prot_id'] = xdf['prot_id'].astype(str)
xdf['residue_index'] = xdf['residue_index'].astype(int)

required = {
    'prot_id',
    'residue_index',
    'y_nuclear',
    'prob_V1',
    'prob_MS',
}

missing = required - set(xdf.columns)

if missing:
    raise RuntimeError(
        f'Wrong/old Nuclear export. Missing columns: {sorted(missing)}; '
        f'got {xdf.columns.tolist()}'
    )

if 'y_small' in xdf.columns:
    raise RuntimeError(
        'WRONG FILE: Nuclear export contains y_small. '
        'Repair NUCLEAR_DEV_OOF first.'
    )

if xdf.duplicated(
    ['prot_id', 'residue_index']
).any():
    raise RuntimeError(
        'Duplicate (prot_id,residue_index) in Nuclear OOF export.'
    )

expected_rows = int(
    train_df.sequence.str.len().sum()
)

if len(xdf) != expected_rows:
    raise RuntimeError(
        f'Nuclear OOF row count mismatch: '
        f'{len(xdf)} vs {expected_rows}'
    )

if set(xdf.prot_id) != set(
    train_df.prot_id.astype(str)
):
    raise RuntimeError(
        'Nuclear OOF protein IDs do not match DevSet1010.'
    )

print('Nuclear export integrity: PASS')
print('rows:', len(xdf))
print('columns:', xdf.columns.tolist())


# ============================================================
# Attach ONLY clean OOF component predictions:
# prob_V1 + prob_MS
#
# Do NOT use prob_FINAL because ensemble weight was selected
# using the full Dev OOF.
# ============================================================

xmap = {
    pid: g.sort_values('residue_index')
    for pid, g in xdf.groupby(
        'prot_id',
        sort=False
    )
}

train_store = ctx.stores[
    STORE_KEY['DevSet1010']
]

for r in train_df.itertuples():

    pid = str(r.prot_id)

    sub = xmap[pid]

    L = len(r.sequence)

    if len(sub) != L:
        raise RuntimeError(
            f'{pid}: Nuclear OOF length '
            f'{len(sub)} != sequence length {L}'
        )

    if not np.array_equal(
        sub.residue_index.to_numpy(),
        np.arange(L)
    ):
        raise RuntimeError(
            f'{pid}: residue_index is not exactly 0..L-1'
        )

    arr = sub[
        [
            'prob_V1',
            'prob_MS',
        ]
    ].to_numpy(
        dtype=np.float32
    )

    if not np.all(
        np.isfinite(arr)
    ):
        raise RuntimeError(
            f'{pid}: non-finite Nuclear OOF probability'
        )

    set_block(
        train_store,
        pid,
        'XTASK_NUCLEAR2_CLEAN',
        arr
    )

print(
    'Attached XTASK_NUCLEAR2_CLEAN: PASS'
)


# ============================================================
# CLEAN DEV REPRESENTATION SCREEN
# ============================================================

BASE_BLOCKS = [
    'ESM',
    'ACCESS',
    'SM_POCKET3D44',
    'SM_TOPO16',
]

GRID = {
    'S_BASE_POCKET_GEOMETRY_CLEANV3':
        BASE_BLOCKS,

    'S_BASE_PLUS_XTASK_NUCLEAR2_CLEANV3':
        BASE_BLOCKS
        + [
            'XTASK_NUCLEAR2_CLEAN'
        ],
}


EVO_OK = bool(
    globals().get(
        'EVO_AVAILABLE',
        False
    )
)


if EVO_OK:

    GRID[
        'S_BASE_PLUS_EVO_CONTROL_CLEANV3'
    ] = (
        BASE_BLOCKS
        + [
            'EVO'
        ]
    )

    GRID[
        'S_BASE_PLUS_EVO_XTASK_NUCLEAR2_CLEANV3'
    ] = (
        BASE_BLOCKS
        + [
            'EVO',
            'XTASK_NUCLEAR2_CLEAN',
        ]
    )


print('\nFresh representations:')

for k, v in GRID.items():
    print(
        ' -',
        k,
        '=',
        v
    )


# IMPORTANT:
# completely new cache namespace -> no stale REUSE
RUN_VERSION = (
    'standalone_small_xtask_screen_'
    'CLEAN_V3_20260914'
)


cv, folds = suite.run_dev_cv_only(
    ctx,
    GRID,
    models=('MLP',),
    run_version=RUN_VERSION,
)


base_name = (
    'S_BASE_POCKET_GEOMETRY_CLEANV3'
)

base = float(
    cv.loc[
        cv.representation.eq(
            base_name
        ),
        'CV_AUPRC'
    ].iloc[0]
)


cv = cv.copy()

cv[
    'delta_vs_base'
] = (
    cv[
        'CV_AUPRC'
    ]
    -
    base
)


cv = (
    cv
    .sort_values(
        'CV_AUPRC',
        ascending=False
    )
    .reset_index(
        drop=True
    )
)


print(
    '\nCLEAN V3 SMALL RESULT'
)

display(
    cv
)

display(
    folds
)


cv.to_csv(
    ADDIN_ROOT
    / 'REPRESENTATION_SCREEN_CLEAN_V3.csv',
    index=False
)

folds.to_csv(
    ADDIN_ROOT
    / 'REPRESENTATION_SCREEN_FOLDS_CLEAN_V3.csv',
    index=False
)


best = cv.iloc[0]


xt = cv[
    cv.representation.eq(
        'S_BASE_PLUS_XTASK_NUCLEAR2_CLEANV3'
    )
]

xt_score = float(
    xt.CV_AUPRC.iloc[0]
)


decision = {

    'run_version':
        RUN_VERSION,

    'base_dev_auprc':
        base,

    'xtask_nuclear2_dev_auprc':
        xt_score,

    'xtask_nuclear2_gain_vs_base':
        xt_score
        -
        base,

    'best_representation':
        str(
            best.representation
        ),

    'best_dev_auprc':
        float(
            best.CV_AUPRC
        ),

    'best_gain_vs_base':
        float(
            best.delta_vs_base
        ),

    'promote_best_if_gain_ge_0.002':
        bool(
            best.delta_vs_base
            >=
            0.002
        ),

    'EVO_control_available':
        EVO_OK,

    'cross_task_source_columns': [
        'prob_V1',
        'prob_MS',
    ],
}


(
    ADDIN_ROOT
    / 'DECISION_CLEAN_V3.json'
).write_text(
    json.dumps(
        decision,
        indent=2
    )
)


print(
    '\nDECISION'
)

print(
    json.dumps(
        decision,
        indent=2
    )
)

print(
    'Saved:',
    ADDIN_ROOT
)

SMALL CLEAN V3 — TRUE NUCLEAR -> SMALL CROSS-TASK SCREEN
Nuclear export integrity: PASS
rows: 170181
columns: ['dataset', 'prot_id', 'residue_index', 'y_nuclear', 'prob_FINAL', 'prob_V1', 'prob_MS']
Attached XTASK_NUCLEAR2_CLEAN: PASS

Fresh representations:
 - S_BASE_POCKET_GEOMETRY_CLEANV3 = ['ESM', 'ACCESS', 'SM_POCKET3D44', 'SM_TOPO16']
 - S_BASE_PLUS_XTASK_NUCLEAR2_CLEANV3 = ['ESM', 'ACCESS', 'SM_POCKET3D44', 'SM_TOPO16', 'XTASK_NUCLEAR2_CLEAN']
 - S_BASE_PLUS_EVO_CONTROL_CLEANV3 = ['ESM', 'ACCESS', 'SM_POCKET3D44', 'SM_TOPO16', 'EVO']
 - S_BASE_PLUS_EVO_XTASK_NUCLEAR2_CLEANV3 = ['ESM', 'ACCESS', 'SM_POCKET3D44', 'SM_TOPO16', 'EVO', 'XTASK_NUCLEAR2_CLEAN']
DONE MLP S_BASE_POCKET_GEOMETRY_CLEANV3 1 AUPRC 0.3846314510628548
DONE MLP S_BASE_POCKET_GEOMETRY_CLEANV3 2 AUPRC 0.4232207851571105
DONE MLP S_BASE_POCKET_GEOMETRY_CLEANV3 3 AUPRC 0.33972522865868254
DONE MLP S_BASE_POCKET_GEOMETRY_CLEANV3 4 AUPRC 0.3516876783465647
DONE MLP S_BASE_POCKET_GEOMETRY_CLEANV3 5 AUPRC 0.38927490858

,model,representation,CV_AUPRC,CV_AUPRC_SD,CV_AUROC,delta_vs_base
0,MLP,S_BASE_PLUS_XTASK_NUCLEAR2_CLEANV3,0.378528,0.031331,0.855914,0.000820
1,MLP,S_BASE_POCKET_GEOMETRY_CLEANV3,0.377708,0.033064,0.849750,0.000000
2,MLP,S_BASE_PLUS_EVO_XTASK_NUCLEAR2_CLEANV3,0.375202,0.025489,0.859751,-0.002506
3,MLP,S_BASE_PLUS_EVO_CONTROL_CLEANV3,0.374507,0.028909,0.854665,-0.003201


,model,representation,fold,AUPRC,AUROC,F1@0.5,MCC@0.5
0,MLP,S_BASE_POCKET_GEOMETRY_CLEANV3,1,0.384631,0.863065,0.386779,0.367907
1,MLP,S_BASE_POCKET_GEOMETRY_CLEANV3,2,0.423221,0.861593,0.415854,0.387586
2,MLP,S_BASE_POCKET_GEOMETRY_CLEANV3,3,0.339725,0.835284,0.362764,0.337324
3,MLP,S_BASE_POCKET_GEOMETRY_CLEANV3,4,0.351688,0.842518,0.367197,0.342158
4,MLP,S_BASE_POCKET_GEOMETRY_CLEANV3,5,0.389275,0.846291,0.395137,0.363943
5,MLP,S_BASE_PLUS_XTASK_NUCLEAR2_CLEANV3,1,0.387025,0.866024,0.375793,0.361304
6,MLP,S_BASE_PLUS_XTASK_NUCLEAR2_CLEANV3,2,0.423581,0.872062,0.362780,0.360378
7,MLP,S_BASE_PLUS_XTASK_NUCLEAR2_CLEANV3,3,0.339640,0.841811,0.336551,0.329429
8,MLP,S_BASE_PLUS_XTASK_NUCLEAR2_CLEANV3,4,0.361078,0.852445,0.371619,0.355046
9,MLP,S_BASE_PLUS_XTASK_NUCLEAR2_CLEANV3,5,0.381318,0.847228,0.374624,0.349525



DECISION
{
  "run_version": "standalone_small_xtask_screen_CLEAN_V3_20260914",
  "base_dev_auprc": 0.3777080103628627,
  "xtask_nuclear2_dev_auprc": 0.3785281696143671,
  "xtask_nuclear2_gain_vs_base": 0.0008201592515043599,
  "best_representation": "S_BASE_PLUS_XTASK_NUCLEAR2_CLEANV3",
  "best_dev_auprc": 0.3785281696143671,
  "best_gain_vs_base": 0.0008201592515043599,
  "promote_best_if_gain_ge_0.002": false,
  "EVO_control_available": true,
  "cross_task_source_columns": [
    "prob_V1",
    "prob_MS"
  ]
}
Saved: /content/drive/MyDrive/Protein-binding-Phi/results/ADDIN_STANDALONE_SMALL_XTASK
